<a href="https://colab.research.google.com/github/Safin44/my-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

**Lane:** Refresh / Content Opportunity Scoring  
**Goal:** Identify content pages whose observed search trend is declining so the content team can prioritize a human review.

This notebook frames the problem; it does **not** train or claim to evaluate a model. I will use the anonymized starter dataset only. One row should represent one content item/page.

## 1. My lane as an ML task (type)

**Task type: Binary classification.**

For each content page, classify whether its observed search-impression trend is declining (`1`) or not declining (`0`). The output can be interpreted as a score for prioritizing pages for human review, but the underlying ML task is classification because the target is a binary label.

**Content action:** Give editors a review queue of pages flagged as declining. The flag is decision support, not proof that content is poor or that an update will improve performance.

**Scope limitation:** The available label describes an observed recent trend, not a guaranteed future decline. I will not describe this as forecasting future performance.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
from IPython.display import display

# Search common locations first. If Colab opened the notebook without the
# repository files, clone your public repository into the Colab runtime.
candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/my-flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None and Path("/content").exists():
    repo_dir = Path("/content/my-flyrank-ml-internship")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Safin44/my-flyrank-ml-internship.git",
             str(repo_dir)],
            check=True,
        )
    candidate_paths.insert(0, repo_dir / "data/raw/content_refresh_anonymized.csv")
    data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Confirm the anonymized starter CSV is present in your repository."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {len(df.columns)}")
display(df.head())

Loaded: /content/my-flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Shape: 30,000 rows × 44 columns
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

**Target:** `is_declining_label`, derived from the observed `trend_direction` column:

- `1` means `trend_direction == "down"`.
- `0` means another observed trend category.

This is an observed-outcome label built from the recent-versus-previous 30-day impression trend. It is not a future outcome. The label's source columns (`trend_direction` and `trend_pct`) must never be used as model features. The component windows that define the trend also need to be excluded from features if they would reveal the label.

I will inspect whether `trend_direction` exists and whether its values are valid before creating the target. The target column is for framing and inspection only; this notebook does not train a model.

In [2]:
# Validate the source column and sketch the target.
if "trend_direction" not in df.columns:
    raise KeyError(
        "Expected 'trend_direction' was not found. Check the repository's data dictionary "
        "and update this cell to match the actual starter dataset."
    )

print("Observed trend categories:")
display(df["trend_direction"].value_counts(dropna=False).rename_axis("trend_direction").to_frame("rows"))

df["is_declining_label"] = df["trend_direction"].eq("down").astype("int8")

print("\nTarget distribution:")
target_summary = (
    df["is_declining_label"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("is_declining_label")
    .to_frame("rows")
)
target_summary["share_pct"] = (target_summary["rows"] / len(df) * 100).round(2)
display(target_summary)

print("\nExample rows with target:")
preview_cols = [c for c in ["content_id", "client_id", "content_type", "trend_direction",
                            "trend_pct", "is_declining_label"] if c in df.columns]
display(df[preview_cols].head(10))

Observed trend categories:


,rows
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Target distribution:


,rows,share_pct
is_declining_label,,
0,13738,45.79
1,16262,54.21



Example rows with target:


,content_id,client_id,content_type,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,keyword article,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,down,-38.9,1
6,content_9a34b442b552,client_8722616204,keyword article,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,keyword article,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,keyword article,down,-29.2,1


## 3. Success metric

**Primary metric: average precision (area under the precision–recall curve, AP / PR-AUC).**

The positive class is a page whose observed trend is declining (`is_declining_label = 1`). Average precision summarizes the precision–recall trade-off across score thresholds and is useful when the team wants to find declining pages without flooding editors with false alarms.

I would compare the model's AP with a simple baseline and report the positive-class prevalence. The data dictionary reports about 54.2% declining labels in the starter slice, so a no-skill ranking has an AP around that prevalence. This is a reference point, not a model result.

This notebook does not train or evaluate a model, so no achieved metric is claimed. Before evaluation, the feature set must exclude label-defining information and validation should respect client grouping.

In [3]:
# Check the observed prevalence to establish the reference level for the metric.
positive_rate = df["is_declining_label"].mean()
print(f"Observed declining-label prevalence: {positive_rate:.3f} ({positive_rate:.1%})")
print(f"Approximate no-skill AP reference:   {positive_rate:.3f}")
print("These are dataset prevalence statistics, not a trained model's performance.")

Observed declining-label prevalence: 0.542 (54.2%)
Approximate no-skill AP reference:   0.542
These are dataset prevalence statistics, not a trained model's performance.


# Confirm the row grain using the actual loaded dataframe.
if "content_id" in df.columns:
    print("Rows:", len(df))
    print("Unique content_id values:", df["content_id"].nunique(dropna=False))
    print("Duplicate content_id rows:", int(df["content_id"].duplicated().sum()))
else:
    print("content_id is not present; inspect the data dictionary to identify the row key.")

if "client_id" in df.columns:
    print("Unique client_id values:", df["client_id"].nunique(dropna=False))

# Display a compact lane-relevant slice: one row per page, with content,
# search-performance, and label columns where present.
lane_columns = [
    "content_id", "client_id", "content_type", "main_intent",
    "content_age_days", "days_since_last_update", "word_count",
    "search_volume", "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "trend_direction", "trend_pct",
    "is_declining_label",
]
lane_columns = [column for column in lane_columns if column in df.columns]
lane_df = df[lane_columns].copy()
print("Lane dataframe shape:", lane_df.shape)
display(lane_df.head(10))

print("\nData types and missing values in lane slice:")
display(lane_df.dtypes.astype(str).rename("dtype").to_frame().join(
    lane_df.isna().sum().rename("missing_values")
))

In [4]:
# Confirm the row grain using the actual loaded dataframe.
if "content_id" in df.columns:
    print("Rows:", len(df))
    print("Unique content_id values:", df["content_id"].nunique(dropna=False))
    print("Duplicate content_id rows:", int(df["content_id"].duplicated().sum()))
else:
    print("content_id is not present; inspect the data dictionary to identify the row key.")

if "client_id" in df.columns:
    print("Unique client_id values:", df["client_id"].nunique(dropna=False))

display(df.head(10))
display(df.dtypes.astype(str).rename("dtype").to_frame().join(
    df.isna().sum().rename("missing_values")
))

Rows: 30000
Unique content_id values: 30000
Duplicate content_id rows: 0
Unique client_id values: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9,1
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2,1


,dtype,missing_values
content_id,object,0
client_id,object,0
search_volume,float64,2468
competition,float64,2468
competition_level,object,2610
cpc,float64,2468
content_type,object,0
main_intent,object,2374
word_count,float64,7699
char_count,float64,7699


## 5. Why ML beats a fixed rule here

A fixed rule might flag every page whose impressions fall below a single threshold. That is easy to explain, but it can miss pages with different search demand, content age, content type, engagement, and visibility patterns.

A classifier can combine multiple non-leaking signals and learn combinations associated with the observed declining label. Its output can help prioritize which pages a person reviews first.

ML is not automatically better. A transparent rule is a useful baseline and may be preferable when labels are weak or the dataset is small. To justify ML, I would compare it with that baseline on held-out clients, using the same metric and a leakage-safe feature set.

**Leakage guard:** `trend_direction` and `trend_pct` define the label and must not be features. Do not use `content_id` or `client_id` as predictive features. The recent/previous impression windows used to calculate the trend also need to be excluded from the feature set for this target. No model is trained in this notebook.

In [5]:
# A small, explicit leakage/identity audit for the proposed feature framing.
label_source_columns = {"trend_direction", "trend_pct"}
identifier_columns = {"content_id", "client_id"}
trend_window_columns = {
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
target_column = "is_declining_label"

print("Target column present:", target_column in df.columns)
print("Label-source columns present:", sorted(label_source_columns.intersection(df.columns)))
print("Identifier columns present:", sorted(identifier_columns.intersection(df.columns)))
print("Trend-window columns present:", sorted(trend_window_columns.intersection(df.columns)))

# These are columns to exclude if/when a feature matrix is designed for this label.
excluded_from_features = sorted(
    (label_source_columns | identifier_columns | trend_window_columns | {target_column})
    .intersection(df.columns)
)
print("\nProposed exclusions from model features:")
print(excluded_from_features)

Target column present: True
Label-source columns present: ['trend_direction', 'trend_pct']
Identifier columns present: ['client_id', 'content_id']
Trend-window columns present: ['clicks_last_30d', 'clicks_prev_30d', 'impressions_last_30d', 'impressions_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']

Proposed exclusions from model features:
['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_label', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']


## 6. Self-check

- [x] Named the ML task type and explained the decision it supports.
- [x] Defined an observed target and distinguished it from a future prediction.
- [x] Selected a defensible metric and stated its baseline reference.
- [x] Loaded the real starter CSV and displayed a DataFrame.
- [x] Explained the unit of analysis and checked the row identifier.
- [x] Sketched the target column using the dataset's label source.
- [x] Explained when ML may help and why a fixed rule remains a baseline.
- [x] Identified label leakage and identifier columns to exclude.
- [ ] Run every cell successfully in the repository/Colab environment and save the executed outputs.
- [ ] Commit this executed notebook to `work/notebooks/w02_ml_task_framing.ipynb`.

**Final claim:** This notebook frames a classification task for content-review prioritization. It does not train a model, prove causal impact, or claim a model score.